# Add the QuSpin Sensors to _3d.py

### To solve long term I need to add the sensor-type to MNE

In [ ]:
# We start by importing the relevant packages
import mne
import importlib.util
from pathlib import Path
import re
import shutil
import time

# 1) Locate _3d.py
spec = importlib.util.find_spec("mne.viz._3d")
if spec is None or spec.origin is None:
    raise RuntimeError("Could not find mne.viz._3d in this Python environment/kernel.")

p = Path(spec.origin)
print("Found:", p)

# 2) Backup
backup = p.with_suffix(p.suffix + f".bak_{time.strftime('%Y%m%d_%H%M%S')}")
shutil.copy2(p, backup)
print("Backup written:", backup)

txt = p.read_text(encoding="utf-8")

# 3) Find the exact elif-block and add the new constant
block_pat = re.compile(
    r"(elif\s+id_\s+in\s*\(\s*"
    r"[\s\S]*?FIFF\.FIFFV_COIL_KIT_REF_MAG[\s\S]*?"
    r"\)\s*:\s*)"
)

m = block_pat.search(txt)
if not m:
    raise RuntimeError(
        "Couldn't find the target 'elif id_ in (...)' block containing FIFFV_COIL_KIT_REF_MAG. "
        "Open the file and confirm the exact text still matches."
    )

block = m.group(0)

new_const = "        FIFF.FIFFV_COIL_QUSPIN_ZFOPM_MAG2,\n"
if "FIFF.FIFFV_COIL_QUSPIN_ZFOPM_MAG2" in block:
    print("Already present — no change needed.")
else:
    # Insert right before the closing "):"
    block_new = re.sub(r"\n\s*\)\s*:\s*$", "\n" + new_const + "    ):",
                       block, flags=re.MULTILINE)
    txt_new = txt[:m.start()] + block_new + txt[m.end():]
    p.write_text(txt_new, encoding="utf-8")
    print("✅ Patched _3d.py (added FIFFV_COIL_QUSPIN_ZFOPM_MAG2).")

# 4) Reminder
print("\nNow restart:")
print("• If you are running a script: stop it and rerun it.")
print("• If you're in a notebook: restart the kernel (VS Code: Kernel → Restart).")
print("• If you imported mne already: restart is required so Python reloads the modified module.")


Using matplotlib as 2D backend.
